# Lesson 4: Vector Databases

**Objective**: Build a retrieval system that efficiently searches for relevant document chunks.

**Topics**:
- Sparse vs. dense retrieval methods
- Hybrid search methods (e.g., combining BM25 with dense retrieval)
- Overview of vector databases: Milvus, Faiss, Qdrant

**Practical Task**: Set up a vector database and implement a retrieval method.

**Resources**:
- What is a vector database
- Choosing a vector database


# 1. Vector Store from Scratch

In [1]:
#!wget --user-agent "Mozilla" "https://arxiv.org/pdf/2307.09288.pdf" -O "data/llama2.pdf"

## Imports

In [81]:
from typing import  Any, cast

from llama_index.core import VectorStoreIndex
from llama_index.core.bridge.pydantic import Field
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.schema import TextNode, BaseNode
from llama_index.core.vector_stores import MetadataFilters, VectorStoreQuery, VectorStoreQueryResult
from llama_index.core.vector_stores.types import BasePydanticVectorStore
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.huggingface import HuggingFaceLLM
from llama_index.readers.file import PyMuPDFReader

import numpy as np


## Setup

We load in some documents, and parse them into `Node` objects - chunks that are ready to be inserted into a vector store.

### Load in Documents

In [82]:
loader = PyMuPDFReader()
documents = loader.load(file_path="./data/llama2.pdf")

In [84]:
len(documents)

77

In [85]:
documents

[Document(id_='9189fcf0-aa76-4b68-8c32-1197fc3b56a6', embedding=None, metadata={'total_pages': 77, 'file_path': './data/llama2.pdf', 'source': '1'}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text_resource=MediaResource(embeddings=None, data=None, text='Llama 2: Open Foundation and Fine-Tuned Chat Models\nHugo Touvron∗\nLouis Martin†\nKevin Stone†\nPeter Albert Amjad Almahairi Yasmine Babaei Nikolay Bashlykov Soumya Batra\nPrajjwal Bhargava Shruti Bhosale Dan Bikel Lukas Blecher Cristian Canton Ferrer Moya Chen\nGuillem Cucurull David Esiobu Jude Fernandes Jeremy Fu Wenyin Fu Brian Fuller\nCynthia Gao Vedanuj Goswami Naman Goyal Anthony Hartshorn Saghar Hosseini Rui Hou\nHakan Inan Marcin Kardas Viktor Kerkez Madian Khabsa Isabel Kloumann Artem Korenev\nPunit Singh Koura Marie-Anne Lachaux Thibaut Lavril Jenya Lee Diana Liskovich\nYinghai Lu Yuning Mao Xavier Martinet Todor Mihaylov Push

### Parse into Nodes

In [86]:
node_parser = SentenceSplitter(chunk_size=256)
nodes = node_parser.get_nodes_from_documents(documents)

In [87]:
nodes

[TextNode(id_='77610012-849b-45b2-89d0-6dfe2d5ee3d4', embedding=None, metadata={'total_pages': 77, 'file_path': './data/llama2.pdf', 'source': '1'}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='9189fcf0-aa76-4b68-8c32-1197fc3b56a6', node_type=<ObjectType.DOCUMENT: '4'>, metadata={'total_pages': 77, 'file_path': './data/llama2.pdf', 'source': '1'}, hash='3c530f8d6ddfeba9dbf51daf1750b8b70891492bbe0a5e1af5a7411220288789'), <NodeRelationship.NEXT: '3'>: RelatedNodeInfo(node_id='806f9cd2-311e-4b5e-83b8-bb9767aca3a9', node_type=<ObjectType.TEXT: '1'>, metadata={}, hash='0fad4e33b55dd637c14c97ea0345801341c96ec5311e77703d4a2e8575d5c471')}, metadata_template='{key}: {value}', metadata_separator='\n', text='Llama 2: Open Foundation and Fine-Tuned Chat Models\nHugo Touvron∗\nLouis Martin†\nKevin Stone†\nPeter Albert Amjad Almahairi Yasmine Babaei Nikolay Bashlykov Soumya Batra\nPrajjwal Bhargava Shruti Bhos

In [88]:
len(nodes)

1405

### Generate Embeddings for each Node

In [89]:
embed_model  = HuggingFaceEmbedding(model_name="BAAI/bge-small-en")

2025-11-07 20:34:28,433 - INFO - Load pretrained SentenceTransformer: BAAI/bge-small-en
2025-11-07 20:34:33,158 - INFO - 1 prompt is loaded, with the key: query


In [90]:
for node in nodes:
    node_embedding = embed_model.get_text_embedding(
        node.get_content(metadata_mode="all")
    )
    node.embedding = node_embedding

In [91]:
node

TextNode(id_='8fea48e0-a8a7-40c1-9845-47ae79bedbab', embedding=[-0.0690780058503151, -0.017995158210396767, 0.014966516755521297, 0.0011772301513701677, 0.03173435479402542, 0.018945416435599327, 0.01059022918343544, 0.006581105757504702, 0.0068051484413445, -0.01549868006259203, 0.0034487613011151552, -0.036508359014987946, 0.03857961669564247, 0.03904419019818306, 0.005557497963309288, 0.04150063171982765, 0.010524988174438477, -0.043533314019441605, 0.0213011521846056, 0.03920348733663559, 0.003421988571062684, 0.02251451089978218, -0.005932754371315241, -0.040764350444078445, 0.013485781848430634, 0.03326113894581795, -0.04733029380440712, -0.022442616522312164, -0.018650848418474197, -0.22758308053016663, -0.021911675110459328, -0.018548594787716866, 0.008822284638881683, 4.955143231200054e-05, -0.04927002638578415, -0.009225321933627129, 0.010113006457686424, 0.0036335685290396214, -0.007135758176445961, 0.043485939502716064, -0.0033680591732263565, 0.020419077947735786, -0.01236

## Build a Simple In-Memory Vector Store

Now we'll build our in-memory vector store. We'll store Nodes within a simple Python dictionary. We'll start off implementing embedding search, and add metadata filters.

## 1. Defining the Interface
We'll first define the interface for building a vector store. It contains the following items:
    - get
    - add
    - delete
    - query
    - persist (which we will not implement)

In [6]:
class BaseVectorStore(BasePydanticVectorStore):
    """Simple custom Vector Store.

    Stores documents in a simple in-memory dict.

    """

    stores_text: bool = True

    def client(self) -> Any:
        """Get client."""
        return None

    def get(self, text_id: str) -> list[float]:
        """Get embedding."""
        pass

    def add(
        self,
        nodes: list[BaseNode],
    ) -> list[str]:
        """Add nodes to index."""
        pass

    def delete(self, ref_doc_id: str, **delete_kwargs: Any) -> None:
        """
        Delete nodes using with ref_doc_id.

        Args:
            ref_doc_id (str): The doc_id of the document to delete.

        """
        pass

    def query(
        self,
        query: VectorStoreQuery,
        **kwargs: Any,
    ) -> VectorStoreQueryResult:
        """Get nodes for response."""
        pass

    def persist(self, persist_path, fs=None) -> None:
        """Persist the SimpleVectorStore to a directory.

        NOTE: we are not implementing this for now.

        """
        pass


At a high-level, we subclass our base `VectorStore` abstraction. There's no inherent reason to do this if you're just building a vector store from scratch. We do it because it makes it easy to plug into our downstream abstractions later.

Let's look at some of the classes defined here.

`BaseNode` is simply the parent class of our core Node modules. Each Node represents a text chunk + associated metadata.
We also use some lower-level constructs, for instance our `VectorStoreQuery` and `VectorStoreQueryResult`. These are just lightweight dataclass containers to represent queries and results. We look at the dataclass fields below.

## 2. Defining add, get, and delete
We add some basic capabilities to add, get, and delete from a vector store.

The implementation is very simple (everything is just stored in a python dictionary).

In [92]:
class VectorStore2(BaseVectorStore):
    """VectorStore2 (add/get/delete implemented)."""

    stores_text: bool = True
    node_dict: dict[str, BaseNode] = Field(default_factory=dict)

    def get(self, text_id: str) -> list[float]:
        """Get embedding."""
        return self.node_dict[text_id]

    def add(
        self,
        nodes: list[BaseNode],
    ) -> list[str]:
        """Add nodes to index."""
        for node in nodes:
            self.node_dict[node.node_id] = node

    def delete(self, node_id: str, **delete_kwargs: Any) -> None:
        """
        Delete nodes using with node_id.

        Args:
            node_id: str

        """
        del self.node_dict[node_id]

In [93]:
test_node = TextNode(id_="id1", text="hello world")
test_node2 = TextNode(id_="id2", text="foo bar")
test_nodes = [test_node, test_node2]

In [94]:
vector_store = VectorStore2()

vector_store.add(test_nodes)

In [95]:
node = vector_store.get("id1")
print(str(node))

Node ID: id1
Text: hello world


## 3.a Defining query (semantic search)
We implement a basic version of top-k semantic search. This simply iterates through all document embeddings, and compute cosine-similarity with the query embedding. The top-k documents by cosine similarity are returned.

Cosine similarity: $\dfrac{\vec{d}\vec{q}}{|\vec{d}||\vec{q}|}$ for every document, query embedding pair $\vec{d}$, $\vec{p}$.

NOTE: The top-k value is contained in the VectorStoreQuery container.

NOTE: Similar to the above, we define another subclass just so we don't have to reimplement the above functions (not because this is actually good code practice).

In [11]:
def get_top_k_embeddings(
    query_embedding: list[float],
    doc_embeddings: list[list[float]],
    doc_ids: list[str],
    similarity_top_k: int = 5,
) -> tuple[list[float], list]:
    """Get top nodes by similarity to the query."""
    # dimensions: D
    q_embed_np = np.array(query_embedding)
    # dimensions: N x D
    d_embed_np = np.array(doc_embeddings)
    # dimensions: N
    d_product_arr = np.dot(d_embed_np, q_embed_np)
    # dimensions: N
    norm_arr = np.linalg.norm(q_embed_np) * np.linalg.norm(
        d_embed_np, axis=1, keepdims=False
    )
    # dimensions: N
    cos_sim_arr = d_product_arr / norm_arr

    # now we have the N cosine similarities for each document
    # sort by top k cosine similarity, and return ids
    tups = [(cos_sim_arr[i], doc_ids[i]) for i in range(len(doc_ids))]
    sorted_tups = sorted(tups, key=lambda t: t[0], reverse=True)

    sorted_tups = sorted_tups[:similarity_top_k]

    result_similarities = [s for s, _ in sorted_tups]
    result_ids = [n for _, n in sorted_tups]
    return result_similarities, result_ids

In [96]:
class VectorStore3A(VectorStore2):
    """Implements semantic/dense search."""

    def query(
        self,
        query: VectorStoreQuery,
        **kwargs: Any,
    ) -> VectorStoreQueryResult:
        """Get nodes for response."""

        query_embedding = cast(list[float], query.query_embedding)
        doc_embeddings = [n.embedding for n in self.node_dict.values()]
        doc_ids = [n.node_id for n in self.node_dict.values()]

        similarities, node_ids = get_top_k_embeddings(
            query_embedding,
            doc_embeddings,
            doc_ids,
            similarity_top_k=query.similarity_top_k,
        )
        result_nodes = [self.node_dict[node_id] for node_id in node_ids]

        return VectorStoreQueryResult(
            nodes=result_nodes, similarities=similarities, ids=node_ids
        )

## 3.b. Supporting Metadata Filtering
The next extension is adding metadata filter support. This means that we will first filter the candidate set with documents that pass the metadata filters, and then perform semantic querying.

For simplicity we use metadata filters for exact matching with an AND condition.

In [97]:
def filter_nodes(nodes: list[BaseNode], filters: MetadataFilters):
    filtered_nodes = []
    for node in nodes:
        matches = True
        for f in filters.filters:
            if f.key not in node.metadata:
                matches = False
                continue
            if f.value != node.metadata[f.key]:
                matches = False
                continue
        if matches:
            filtered_nodes.append(node)
    return filtered_nodes

We add filter_nodes as a first-pass over the nodes before running semantic search.

In [98]:
def dense_search(query: VectorStoreQuery, nodes: list[BaseNode]):
    """Dense search."""
    query_embedding = cast(list[float], query.query_embedding)
    doc_embeddings = [n.embedding for n in nodes]
    doc_ids = [n.node_id for n in nodes]

    return get_top_k_embeddings(
        query_embedding,
        doc_embeddings,
        doc_ids,
        similarity_top_k=query.similarity_top_k,
    )

In [15]:
class VectorStore3B(VectorStore2):
    """Implements Metadata Filtering."""

    def query(
        self,
        query: VectorStoreQuery,
        **kwargs: Any,
    ) -> VectorStoreQueryResult:
        """Get nodes for response."""
        # 1. First filter by metadata
        nodes = self.node_dict.values()
        if query.filters is not None:
            nodes = filter_nodes(nodes, query.filters)
        if len(nodes) == 0:
            result_nodes = []
            similarities = []
            node_ids = []
        else:
            # 2. Then perform semantic search
            similarities, node_ids = dense_search(query, nodes)
            result_nodes = [self.node_dict[node_id] for node_id in node_ids]
        return VectorStoreQueryResult(
            nodes=result_nodes, similarities=similarities, ids=node_ids
        )

## 4. Load Data into our Vector Store
Let's load our text chunks into the vector store, and run it on different types of queries: dense search, w/ metadata filters, and more.

In [99]:
vector_store = VectorStore3B()
# load data into the vector stores
vector_store.add(nodes)

Define an example question and embed it.

In [100]:
query_str = "Can you tell me about the key concepts for safety finetuning"
query_embedding = embed_model.get_query_embedding(query_str)
print(query_embedding)

[-0.057459719479084015, 0.0006168404361233115, 0.007399792782962322, -0.02610773779451847, 0.010598315857350826, 0.00798750203102827, 0.06677503883838654, 0.03960995748639107, 0.0025795777328312397, -0.00429012905806303, -0.051723357290029526, -0.036634426563978195, 0.01641755737364292, 0.020253067836165428, 0.0051087080501019955, 0.003774326993152499, 0.032964076846838, 0.032009292393922806, -0.008439945057034492, 0.019312605261802673, -0.004506593104451895, 0.03124888800084591, -0.024012679234147072, -0.020971687510609627, 0.00496076513081789, -0.0026956028304994106, 0.017490491271018982, 0.002253093058243394, -0.017213137820363045, -0.19273661077022552, -0.04692847281694412, -0.03517276048660278, -0.02127939835190773, -0.007309691980481148, -0.02094404771924019, -0.008686757646501064, -0.02881697379052639, 0.031263578683137894, -0.004988396540284157, -0.003774901619181037, 0.04736080393195152, 0.017691638320684433, -0.008748934604227543, -0.009141998365521431, -0.01443556509912014, 

### Query the vector store with dense search.

In [101]:
query_obj = VectorStoreQuery(
    query_embedding=query_embedding, similarity_top_k=2
)

query_result = vector_store.query(query_obj)
for similarity, node in zip(query_result.similarities, query_result.nodes):
    print(
        "\n----------------\n"
        f"[Node ID {node.node_id}] Similarity: {similarity}\n\n"
        f"{node.get_content(metadata_mode='all')}"
        "\n----------------\n\n"
    )


----------------
[Node ID 1bbf2cd8-1411-4f16-b3ac-f537cb77e301] Similarity: 0.8711294993852078

total_pages: 77
file_path: ./data/llama2.pdf
source: 23

Benchmarks give a summary view of model capabilities and behaviors that allow us to understand general
patterns in the model, but they do not provide a fully comprehensive view of the impact the model may have
on people or real-world outcomes; that would require study of end-to-end product deployments. Further
testing and mitigation should be done to understand bias and other social issues for the specific context
in which a system may be deployed. For this, it may be necessary to test beyond the groups available in
the BOLD dataset (race, religion, and gender). As LLMs are integrated and deployed, we look forward to
continuing research that will amplify their potential for positive impact on these important social issues.
4.2
Safety Fine-Tuning
In this section, we describe our approach to safety fine-tuning, including safety categori

### Query the vector store with dense search + Metadata Filters

In [102]:
# filters = MetadataFilters(
#     filters=[
#         ExactMatchFilter(key="page", value=3)
#     ]
# )
filters = MetadataFilters.from_dict({"source": "24"})

query_obj = VectorStoreQuery(
    query_embedding=query_embedding, similarity_top_k=2, filters=filters
)

query_result = vector_store.query(query_obj)
for similarity, node in zip(query_result.similarities, query_result.nodes):
    print(
        "\n----------------\n"
        f"[Node ID {node.node_id}] Similarity: {similarity}\n\n"
        f"{node.get_content(metadata_mode='all')}"
        "\n----------------\n\n"
    )


----------------
[Node ID adcb24c1-8051-4df2-86d7-2111c3d6fa1a] Similarity: 0.8589715396624741

total_pages: 77
file_path: ./data/llama2.pdf
source: 24

We then define best practices for safe and helpful model responses: the model should first address immediate
safety concerns if applicable, then address the prompt by explaining the potential risks to the user, and finally
provide additional information if possible. We also ask the annotators to avoid negative user experience
categories (see Appendix A.5.2). The guidelines are meant to be a general guide for the model and are
iteratively refined and revised to include newly identified risks.
4.2.2
Safety Supervised Fine-Tuning
In accordance with the established guidelines from Section 4.2.1, we gather prompts and demonstrations
of safe model responses from trained annotators, and use the data for supervised fine-tuning in the same
manner as described in Section 3.1. An example can be found in Table 5.
The annotators are instructed to 

/var/folders/gp/286d79g11513f730wt0d61kc0000gq/T/ipykernel_34123/4088133968.py:6: DeprecationWarning: Call to deprecated class method from_dict. (`from_dict()` is deprecated. Please use `MetadataFilters(filters=.., condition='and')` directly instead.)
  filters = MetadataFilters.from_dict({"source": "24"})


## Build a RAG System with the Vector Store
Now that we've built the RAG system, it's time to plug it into our downstream system!

In [103]:
index = VectorStoreIndex.from_vector_store(vector_store, embed_model=embed_model)

In [104]:
llm = HuggingFaceLLM(
    model_name="google/gemma-3-270m", 
    tokenizer_name="google/gemma-3-270m",
    device_map="auto"
)

In [105]:
response = llm.complete("What is the meaning of life?")
print(str(response))

 Is it “good or bad”, “easy or hard”? All answers are not available in a day. Even if we were to answer all of those questions, then we would get a whole lot of unanswered questions.

The answer to that question, I think, is actually quite simple: “good”.

I think there’s a lot of truth to that answer. I’ve been in some of the most hectic, stressful, and exhausting jobs of my life since I was a kid, so having a “good” job – for that matter, “good” life – is absolutely essential.

But I’ve never really understood why it’s important to have a “good” job. Why does a “good” life matter at all? Let’s examine some of the reasons:

<strong>1. Wealth.</strong> “Wealth” seems to be the buzzword today. Wealthy people are said to have “great” and “good” lives. But that’s just “false, stupid, and self-defeating.” It’s not.

To understand the meaning of “wealth”, you have to understand what “wealth” really means. And that’s where the world as we know it could be based.

My father was


In [106]:
query_engine = index.as_query_engine(llm=llm)

In [107]:
query_str = "Can you tell me about the key concepts for safety finetuning"

In [109]:
response = query_engine.query(query_str)
print(str(response))


Our approach to finetuning safety has four distinct parts.
1. Prompt/Safe Demonstration
a. Prompt: We identify and create a safe demonstration to use in the guidance to answer the
question to the best of our ability.
b. Validation: We use a video demonstration to assess the safety of the prompt. We use
the following principles to evaluate the safety of the demonstration:
387In addition to showing the demonstration, we evaluate the feedback. A negative feedback is
an example of safety bias, so we take these as feedback on the safe demonstration and make
sure it is clear as possible what the guidance should be so we can improve. We also use this
to confirm what was observed and how it changed our recommendations in light of
feedback.
2. Guidance/Feedback
a. Guidance: We use a visual demonstration of the prompt. This is to help us identify how
the prompt should be implemented and to assess how well the prompt is being
implemented.
b. Feedback: We also use a visual demonstration of the gu

## Conclusion
That's it! We've built a simple in-memory vector store that supports very simple inserts, gets, deletes, and supports dense search and metadata filtering. This can then be plugged into the rest of LlamaIndex abstractions.

It doesn't support sparse search yet and is obviously not meant to be used in any sort of actual app. But this should expose some of what's going on under the hood!

# 2. Vector Store Using Libraries

#### Load the dataset

In [1]:
from langchain_community.document_loaders import PyPDFLoader
from dotenv import load_dotenv

load_dotenv()

file_path = (
   "data/visual_instruction_tunning.pdf"
)
loader = PyPDFLoader(file_path)
docs = loader.load_and_split()

/Users/felipecisternas/Desktop/uc/practicos-rag/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
docs[0].page_content

'Visual Instruction Tuning\nHaotian Liu1∗, Chunyuan Li2∗, Qingyang Wu3, Yong Jae Lee1\n1University of Wisconsin–Madison 2Microsoft Research 3Columbia University\nhttps://llava-vl.github.io\nAbstract\nInstruction tuning large language models (LLMs) using machine-generated\ninstruction-following data has been shown to improve zero-shot capabilities on\nnew tasks, but the idea is less explored in the multimodal field. We present the\nfirst attempt to use language-only GPT-4 to generate multimodal language-image\ninstruction-following data. By instruction tuning on such generated data, we in-\ntroduce LLaV A:Large Language and Vision Assistant, an end-to-end trained\nlarge multimodal model that connects a vision encoder and an LLM for general-\npurpose visual and language understanding. To facilitate future research on visual\ninstruction following, we construct two evaluation benchmarks with diverse and\nchallenging application-oriented tasks. Our experiments show that LLaV A demon-\nstra

### Embeddings function

In [3]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")
embedded_document = embedding_model.embed_query(docs[0].page_content)
embedded_document[:3]

[0.027461927384138107, 0.04477216303348541, -0.015285043977200985]

# A first approach

In [4]:
from dotenv import load_dotenv

load_dotenv()

True

In [5]:
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams

In [6]:
client = QdrantClient(path="/tmp/langchain_qdrant")

In [7]:
client.delete_collection(collection_name="demo_collection")

True

In [8]:
client.create_collection(
    collection_name="demo_collection",
    vectors_config=VectorParams(size=768, distance=Distance.COSINE),
)

vector_store = QdrantVectorStore(
    client=client,
    collection_name="demo_collection",
    embedding=embedding_model,
)

In [9]:
vector_store.add_documents(docs)


['3249699a5c314d7981090d62df689a22',
 '81ed52e263b24ea2b0f992c45a04df85',
 'f18e0c80220c49b5b24e784f8c74a96f',
 '5732c2337da74c6fbdfddc659f4225b0',
 'af5e6a57606e473ea33e2951e4df628e',
 'a07989ba6da34217a37133822eeb07ef',
 'ee681869a3cf4db2ae7292f2ce1985c2',
 '5f8eb0c5f5cc4955aa0853510a5a4824',
 '2b4219d9d4134de5898465167fbf7960',
 'f52162e048c645fcaf77e127acfbc6ae',
 'd78513d0745c4d2798c042ac3acc1509',
 '005d4d45d241407c9fc70596fef05540',
 'f928fc53dc1444e78c17264d2f39b105',
 'bd4f61b13a304b9d9c370a0daaec05dd',
 '84d56ca06c5c40e899b307d671510e5c',
 '530bed253f394aca8d144964338454eb',
 '9c2fe8542a034a029fd6ea444e31a7e6',
 'e8a97a4aa7a045129820271d1e2dd22b',
 '1b9900cb19b74da6bf529eb8d521bdf2',
 '23b1867afba541ddb5583674698b99f8',
 '53e3620cef94469ca322d60830467925',
 'b5e9672361c3404db7672b0ba9ee1bee',
 '88a1dd50fb9f43adb062b87fbba0538a',
 '967e64c0f3cc4c699f3c2037359af032',
 'a3cdca4e91f94f81af5d2b08554a65e7',
 '0aa43afacf7a412fa170902cce053dd3',
 'd4584902492240a5af298dd8d11760fd',
 

In [11]:
client.scroll(collection_name="demo_collection", limit=10)

([Record(id='005d4d45d241407c9fc70596fef05540', payload={'page_content': 'fridge example (right), perceiving the correct brand of the yogurt requires the model to process high\nresolution images and possess extensive knowledge coverage. We also observed an interesting failure\nof LLaV A, as it responds withyes when asked if strawberry-flavored yogurt is present, even though\nthe fridge contains only yogurt and strawberries. This indicates that, at times, LLaV A perceives\nthe image as a “bag of patches”, failing to grasp the complex semantics within the image. We hope\nLLaV A serves as a solid baseline on the benchmarks, on which our findings can inspire future work\nin developing more capable LMMs.\n7', 'metadata': {'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-12-14T01:02:53+00:00', 'author': '', 'keywords': '', 'moddate': '2023-12-14T01:02:53+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea v

# Dense search

In [12]:
from langchain_qdrant import RetrievalMode

qdrant = QdrantVectorStore.from_documents(
    docs,
    embedding=embedding_model,
    location=":memory:",
    collection_name="my_documents",
    retrieval_mode=RetrievalMode.DENSE,
)

In [13]:
query = "What datasets they used to benchmark LLAVA?"
found_docs = qdrant.similarity_search(query)

In [14]:
found_docs

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-12-14T01:02:53+00:00', 'author': '', 'keywords': '', 'moddate': '2023-12-14T01:02:53+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'data/visual_instruction_tunning.pdf', 'total_pages': 25, 'page': 19, 'page_label': '20', '_id': '42563df178b6496789f942bad6571a8f', '_collection_name': 'my_documents'}, page_content='C Training Details\nWe pre-train our model on the filtered CC-595K subset for 1 epoch with a learning rate of 2e-3 and a\nbatch size of 128, and fine-tune on the proposed LLaV A-Instruct-158K dataset for 3 epochs, with a\nlearning rate of 2e-5 and a batch size of 32. Following Vicuna, we use the Adam optimizer with no\nweight decay and a cosine learning rate with a warmup ratio of 3%. During finetuning, FSDP (Full\nShard Data Parallel) and gradi

In [15]:
retriever = qdrant.as_retriever()
retriever.invoke(query)

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-12-14T01:02:53+00:00', 'author': '', 'keywords': '', 'moddate': '2023-12-14T01:02:53+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'data/visual_instruction_tunning.pdf', 'total_pages': 25, 'page': 19, 'page_label': '20', '_id': '42563df178b6496789f942bad6571a8f', '_collection_name': 'my_documents'}, page_content='C Training Details\nWe pre-train our model on the filtered CC-595K subset for 1 epoch with a learning rate of 2e-3 and a\nbatch size of 128, and fine-tune on the proposed LLaV A-Instruct-158K dataset for 3 epochs, with a\nlearning rate of 2e-5 and a batch size of 32. Following Vicuna, we use the Adam optimizer with no\nweight decay and a cosine learning rate with a warmup ratio of 3%. During finetuning, FSDP (Full\nShard Data Parallel) and gradi

# Sparse Vector Search

To search with only sparse vectors,

The retrieval_mode parameter should be set to RetrievalMode.SPARSE.
An implementation of the SparseEmbeddings interface using any sparse embeddings provider has to be provided as value to the sparse_embedding parameter.
The langchain-qdrant package provides a FastEmbed based implementation out of the box.

In [16]:
from langchain_qdrant import FastEmbedSparse, RetrievalMode

sparse_embeddings = FastEmbedSparse(model_name="Qdrant/bm25", cache_dir="cache")

qdrant = QdrantVectorStore.from_documents(
    docs,
    embedding=embedding_model,
    sparse_embedding=sparse_embeddings,
    location=":memory:",
    collection_name="my_documents",
    retrieval_mode=RetrievalMode.SPARSE,
)

In [17]:
found_docs = qdrant.similarity_search(query)

In [18]:
found_docs

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-12-14T01:02:53+00:00', 'author': '', 'keywords': '', 'moddate': '2023-12-14T01:02:53+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'data/visual_instruction_tunning.pdf', 'total_pages': 25, 'page': 19, 'page_label': '20', '_id': '93634f3ed82943fab95edef24ca3a56f', '_collection_name': 'my_documents'}, page_content='C Training Details\nWe pre-train our model on the filtered CC-595K subset for 1 epoch with a learning rate of 2e-3 and a\nbatch size of 128, and fine-tune on the proposed LLaV A-Instruct-158K dataset for 3 epochs, with a\nlearning rate of 2e-5 and a batch size of 32. Following Vicuna, we use the Adam optimizer with no\nweight decay and a cosine learning rate with a warmup ratio of 3%. During finetuning, FSDP (Full\nShard Data Parallel) and gradi

# Hybrid Search

To perform a hybrid search using dense and sparse vectors with score fusion,

The retrieval_mode parameter should be set to RetrievalMode.HYBRID.
A dense embeddings value should be provided to the embedding parameter.
An implementation of the SparseEmbeddings interface using any sparse embeddings provider has to be provided as value to the sparse_embedding parameter.
Note that if you've added documents with the HYBRID mode, you can switch to any retrieval mode when searching. Since both the dense and sparse vectors are available in the collection.

In [19]:
from langchain_qdrant import FastEmbedSparse, RetrievalMode

sparse_embeddings = FastEmbedSparse(model_name="Qdrant/bm25")

qdrant = QdrantVectorStore.from_documents(
    docs,
    embedding=embedding_model,
    sparse_embedding=sparse_embeddings,
    location=":memory:",
    collection_name="my_documents",
    retrieval_mode=RetrievalMode.HYBRID,
)

In [20]:
found_docs = qdrant.similarity_search(query)

In [21]:
found_docs

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-12-14T01:02:53+00:00', 'author': '', 'keywords': '', 'moddate': '2023-12-14T01:02:53+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'data/visual_instruction_tunning.pdf', 'total_pages': 25, 'page': 19, 'page_label': '20', '_id': '04e28d45a856466cabf9ed8d10f54398', '_collection_name': 'my_documents'}, page_content='C Training Details\nWe pre-train our model on the filtered CC-595K subset for 1 epoch with a learning rate of 2e-3 and a\nbatch size of 128, and fine-tune on the proposed LLaV A-Instruct-158K dataset for 3 epochs, with a\nlearning rate of 2e-5 and a batch size of 32. Following Vicuna, we use the Adam optimizer with no\nweight decay and a cosine learning rate with a warmup ratio of 3%. During finetuning, FSDP (Full\nShard Data Parallel) and gradi

In [22]:
#If you want to execute a similarity search and receive the corresponding scores you can run:
results = vector_store.similarity_search_with_score(
    query=query, k=1
)
for doc, score in results:
    print(f"* [SIM={score:3f}] * {doc.page_content} [{doc.metadata}]")

* [SIM=0.445506] * C Training Details
We pre-train our model on the filtered CC-595K subset for 1 epoch with a learning rate of 2e-3 and a
batch size of 128, and fine-tune on the proposed LLaV A-Instruct-158K dataset for 3 epochs, with a
learning rate of 2e-5 and a batch size of 32. Following Vicuna, we use the Adam optimizer with no
weight decay and a cosine learning rate with a warmup ratio of 3%. During finetuning, FSDP (Full
Shard Data Parallel) and gradient checkpointing is used to save GPU memory, and offloading is not
used. BF16 and TF32 are enabled to achieve a balance between speed and precision.
We train all models with 8× A100s. Pretraining on CC-595K completes within 4 hours. Finetuning
on Instruct-158K completes within 10 hours. Finetuning on ScienceQA completes within 4 hours.
D Assets
Our source code, generated instruction-tuning data, proposed benchmark are uploaded to the
anonymized GitHub repository: LLaV A-Annonymous/LLaV A.
1. Source Code: link
2. README: link
3. In

# Metadata filtering

In [24]:
from qdrant_client.http import models

results = vector_store.similarity_search(
    query=query,
    k=1,
)
for doc in results:
    print(f"*[{doc.metadata}]")

*[{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-12-14T01:02:53+00:00', 'author': '', 'keywords': '', 'moddate': '2023-12-14T01:02:53+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'data/visual_instruction_tunning.pdf', 'total_pages': 25, 'page': 19, 'page_label': '20', '_id': '1337a97a7de341da951d585b6920077d', '_collection_name': 'demo_collection'}]


In [25]:
results

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-12-14T01:02:53+00:00', 'author': '', 'keywords': '', 'moddate': '2023-12-14T01:02:53+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'data/visual_instruction_tunning.pdf', 'total_pages': 25, 'page': 19, 'page_label': '20', '_id': '1337a97a7de341da951d585b6920077d', '_collection_name': 'demo_collection'}, page_content='C Training Details\nWe pre-train our model on the filtered CC-595K subset for 1 epoch with a learning rate of 2e-3 and a\nbatch size of 128, and fine-tune on the proposed LLaV A-Instruct-158K dataset for 3 epochs, with a\nlearning rate of 2e-5 and a batch size of 32. Following Vicuna, we use the Adam optimizer with no\nweight decay and a cosine learning rate with a warmup ratio of 3%. During finetuning, FSDP (Full\nShard Data Parallel) and gr

In [28]:
results = vector_store.similarity_search(
    query=query,
    k=1,
    filter=models.Filter(
        must=[
            models.FieldCondition(
                key="metadata.producer",  # Note the "metadata." prefix!
                match=models.MatchValue(
                    value="pdfTeX-1.40.25"
                ),
            ),
        ]
    ),
)
for doc in results:
    print(f"* {doc.page_content} [{doc.metadata}]")

* C Training Details
We pre-train our model on the filtered CC-595K subset for 1 epoch with a learning rate of 2e-3 and a
batch size of 128, and fine-tune on the proposed LLaV A-Instruct-158K dataset for 3 epochs, with a
learning rate of 2e-5 and a batch size of 32. Following Vicuna, we use the Adam optimizer with no
weight decay and a cosine learning rate with a warmup ratio of 3%. During finetuning, FSDP (Full
Shard Data Parallel) and gradient checkpointing is used to save GPU memory, and offloading is not
used. BF16 and TF32 are enabled to achieve a balance between speed and precision.
We train all models with 8× A100s. Pretraining on CC-595K completes within 4 hours. Finetuning
on Instruct-158K completes within 10 hours. Finetuning on ScienceQA completes within 4 hours.
D Assets
Our source code, generated instruction-tuning data, proposed benchmark are uploaded to the
anonymized GitHub repository: LLaV A-Annonymous/LLaV A.
1. Source Code: link
2. README: link
3. Instructions to lau

In [27]:
results


[]

## Query by turning into a retriever

In [70]:
retriever = vector_store.as_retriever(search_type="mmr", search_kwargs={"k": 5})
retriever.invoke(query)

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-12-14T01:02:53+00:00', 'author': '', 'keywords': '', 'moddate': '2023-12-14T01:02:53+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'data/visual_instruction_tunning.pdf', 'total_pages': 25, 'page': 19, 'page_label': '20', '_id': '9c1b515339ff4f05bf5eb8d0045d13f0', '_collection_name': 'demo_collection'}, page_content='C Training Details\nWe pre-train our model on the filtered CC-595K subset for 1 epoch with a learning rate of 2e-3 and a\nbatch size of 128, and fine-tune on the proposed LLaV A-Instruct-158K dataset for 3 epochs, with a\nlearning rate of 2e-5 and a batch size of 32. Following Vicuna, we use the Adam optimizer with no\nweight decay and a cosine learning rate with a warmup ratio of 3%. During finetuning, FSDP (Full\nShard Data Parallel) and gr